### Простое API - **Earthquake**

#### Получаем данные по пяти самым сильным землятрясениям за день, который был месяц назад в виде файла формата **csv**

In [1]:
import requests 
import pandas as pd 
import datetime as dt

# https://earthquake.usgs.gov/fdsnws/event/1/[METHOD[?PARAMETERS]]
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"


# Выгружаем данные из API
response = requests.get(
        url,
        params={
            "format": "geojson",
            "starttime":"NOW - 7 days", # YYYY-MM-DDTHH:mm:ss±HH:mm
            "endtime":"NOW - 6 days",
            "orderby":"magnitude",
            "limit":"5",
        }
)

# Проверяем ошибки
response.raise_for_status()

# Преобразовываем в словарь Python
j_data = response.json()

print(j_data)

# Дата выгрузки данных
load_time = dt.datetime.today()
load_date = load_time.strftime("%Y-%m-%d %H:%M:%S")
print(load_date)

# Создаём пустой список
all_data = []

for list in j_data["features"]: 
    #print(list)
    # Достаём необходимые поля из выгрузки 
    mag        = list["properties"]["mag"]
    place      = list["properties"]["place"]
    event_url  = list["properties"]["url"]
    event_time = list["properties"]["time"]/1000 # переводим в секунды

    # Переводим в формат даты
    date = dt.datetime.fromtimestamp(event_time)
    event_date = date.strftime("%Y-%m-%d %H:%M:%S")

    # Создаём словарь c нужными полями    
    records = {}

    records["mag"] = mag
    records["place"] = place
    records["event_url"] = event_url
    records["event_time"] = event_date
    records["update_at"] = load_date

    #print(records)
    all_data.append(records)

print(all_data)

# Преобразуем справочник в таблицу
df = pd.json_normalize(all_data)

print(f'Кол-во строк: {len(df)}')
print(df)
 
# # Указываем название файла csv
# csv_path = f"max_earthquakes_pk.csv"

# # Сохраняем в формате csv
# df.to_csv(csv_path, index=False, sep=';')


{'type': 'FeatureCollection', 'metadata': {'generated': 1770560136000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=NOW+-+7+days&endtime=NOW+-+6+days&orderby=magnitude&limit=5', 'title': 'USGS Earthquakes', 'status': 200, 'api': '1.14.1', 'limit': 5, 'offset': 1, 'count': 5}, 'features': [{'type': 'Feature', 'properties': {'mag': 5.3, 'place': '196 km W of Riverton, New Zealand', 'time': 1770020141052, 'updated': 1770023759866, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us6000s5x6', 'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000s5x6&format=geojson', 'felt': 3, 'cdi': 2.3, 'mmi': None, 'alert': None, 'status': 'reviewed', 'tsunami': 0, 'sig': 433, 'net': 'us', 'code': '6000s5x6', 'ids': ',us6000s5x6,usauto6000s5x6,', 'sources': ',us,usauto,', 'types': ',dyfi,internal-moment-tensor,moment-tensor,origin,phase-data,', 'nst': 63, 'dmin': 0.952, 'rms': 0.6, 'gap': 62, 'magType': 'mww', 'type': 'earthq